# Sudden drift detection
### Hypothesis: Sudden political/event-driven changes produce detectable drift in news language streams.

### Can drift detectors identify sharp changes corresponding to major political events?

- As a drift detectors we've chosen **ADWIN**, since it is most suitable for abrupt, sudden shifts like presidential change, war beginning etc.
- For mapping words -> numbers we need to choose good feature extractors:
1. **TF-IDF** as a baseline (easier, comprehensible to interpret) = which words appear
2. **Embeddings** as a more advanced feature extractor (stronger, less interpretable) = what words mean

## Experiment design:
1. **Step 1** - gather data from last 2 years using gdelt, trafilatura from more than one newspaper, considering both left and right wing magazines. There will be 2 sources from left and 1 source from right.
    - Left wing sources: "thenation.com", "motherjones.com"
    - Right wing sources: "breitbart.com"
2. **Step 2** - consider a stream: mix of both. 
    - Order articles in chronological order
3. **Step 3** - pick crucial political events:
    - 2024-11-05  US presidential election
    - 2025-01-20  presidential inauguration
    These will serve us "ground truth" event dates.
4. **Step 4** - test 2 feature extracting methods mentioned above: TF-IDF and embeddings
5. **Step 5** - using drift detecting model (ADWIN) try to predict crucial events, based on distribution change in language streams

Articles
→ group by week and stream group
→ create one weekly text representation
→ extract TF-IDF or embedding vector
→ compute cosine distance between consecutive weeks
→ feed distance stream into ADWIN, PageHinkley, KSWIN
→ compare detected drift dates with real political events

In [466]:
from pathlib import Path
import sys
import os

PROJECT_ROOT = Path.cwd()

if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

os.chdir(PROJECT_ROOT)

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print("Current working directory:", Path.cwd())
print("Project root:", PROJECT_ROOT)

Current working directory: c:\Users\krzys\OneDrive\Pulpit\Studia\StudiaDS\2sem\MLDSt\Project\Concept-Drift-Detector
Project root: c:\Users\krzys\OneDrive\Pulpit\Studia\StudiaDS\2sem\MLDSt\Project\Concept-Drift-Detector


In [467]:
import json
from pathlib import Path
from datetime import datetime, timedelta

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_distances

from river import drift

In [468]:
INPUT_FILE = Path("data/old/polarized_combined1.jsonl")

# Restrict analysis window:
# one month before Trump hush money trial begins (2024-04-15)
# and one month after Trump inauguration (2025-01-20)
START_DATE = pd.Timestamp("2024-03-15")
END_DATE = pd.Timestamp("2025-02-20")

OUTPUT_DIR = Path("results/h1")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

TEXT_COLUMN = "text"
DATE_COLUMN = "timestamp"
LABEL_COLUMN = "label"
DOMAIN_COLUMN = "domain"
URL_COLUMN = "url"


LABEL_TO_STREAM = {
    0: "left",
    1: "right",
    # 2 is neutral, but only used in mixed stream
}

# Choose:
# "D" = daily grouping
# "W" = weekly grouping
TIME_WINDOW = "W"

# Recommended:
# daily: 5
# weekly: 10 or 15
MIN_ARTICLES_PER_WINDOW = 5 if TIME_WINDOW == "D" else 15

# Choose:
# "tfidf"
# "embedding"
FEATURE_TYPE = "embedding"

EVENTS = {
    "Iowa caucuses": "2024-01-15",
    "New Hampshire primary": "2024-01-23",
    "Super Tuesday primaries": "2024-03-05",

    "Trump hush money trial begins": "2024-04-15",
    "Trump conviction": "2024-05-30",

    "First presidential debate": "2024-06-27",

    "Trump assassination attempt": "2024-07-13",
    "Republican National Convention": "2024-07-15",

    "Biden withdraws from race": "2024-07-21",
    "Harris becomes Democratic nominee": "2024-08-05",
    "Democratic National Convention": "2024-08-19",

    "Second presidential debate": "2024-09-10",
    "Vice presidential debate": "2024-10-01",

    "US presidential election": "2024-11-05",
    "Election results confirmed": "2024-11-06",

    "Electoral College vote": "2024-12-17",
    "Congress certifies election": "2025-01-06",

    "Trump inauguration": "2025-01-20"
}

# Event matching tolerance.
# Daily windows need smaller tolerance.
# Weekly windows need larger tolerance.
TOLERANCE_DAYS = 7 if TIME_WINDOW == "D" else 14

### Helpers

In [469]:
def window_name():
    return "daily" if TIME_WINDOW == "D" else "weekly"


def get_output_dir(feature_type, time_window, stream_group):
    w_name = "daily" if time_window == "D" else "weekly"

    path = OUTPUT_DIR / feature_type / w_name / stream_group
    path.mkdir(parents=True, exist_ok=True)

    return path


def get_global_output_dir(feature_type, time_window):
    w_name = "daily" if time_window == "D" else "weekly"

    path = OUTPUT_DIR / feature_type / w_name / "_global"
    path.mkdir(parents=True, exist_ok=True)

    return path

### Data loading

In [470]:
def load_jsonl(path: Path) -> pd.DataFrame:
    rows = []

    if not path.exists():
        raise FileNotFoundError(f"Input file not found: {path}")

    with path.open("r", encoding="utf-8") as f:
        for line in f:
            try:
                rows.append(json.loads(line))
            except json.JSONDecodeError:
                continue

    df = pd.DataFrame(rows)

    if df.empty:
        raise ValueError("Input file is empty or could not be read.")

    required_columns = [
        DATE_COLUMN,
        DOMAIN_COLUMN,
        LABEL_COLUMN,
        TEXT_COLUMN,
        URL_COLUMN,
    ]

    missing = [col for col in required_columns if col not in df.columns]

    if missing:
        raise ValueError(f"Missing required columns: {missing}")

    # Parse dates robustly and make them timezone-naive for easier filtering/plotting.
    df["date"] = pd.to_datetime(df[DATE_COLUMN], errors="coerce", utc=True).dt.tz_convert(None)
    df = df.dropna(subset=["date", TEXT_COLUMN])

    # Restrict the stream to the selected experiment period.
    df = df[(df["date"] >= START_DATE) & (df["date"] <= END_DATE)]

    df[TEXT_COLUMN] = df[TEXT_COLUMN].astype(str)
    df = df[df[TEXT_COLUMN].str.len() > 300]

    df = df.sort_values("date").reset_index(drop=True)

    return df


### Time window aggregation

In [471]:
def get_window_start(series: pd.Series, time_window: str) -> pd.Series:
    if time_window == "D":
        return series.dt.floor("D")

    if time_window == "W":
        return series.dt.to_period("W").apply(lambda r: r.start_time)

    raise ValueError("TIME_WINDOW must be 'D' or 'W'")


def aggregate_stream_time_windows(
    df: pd.DataFrame,
    time_window: str = TIME_WINDOW,
    min_articles_per_window: int = MIN_ARTICLES_PER_WINDOW,
) -> pd.DataFrame:
    """
    Creates 3 streams:
        mixed = all articles, including neutral
        left  = label 0 only
        right = label 1 only
    """

    df = df.copy()
    df["time_window"] = get_window_start(df["date"], time_window)

    mixed_df = df.copy()
    mixed_df["stream_group"] = "mixed"

    ideological_df = df.copy()
    ideological_df["stream_group"] = ideological_df[LABEL_COLUMN].map(LABEL_TO_STREAM)
    ideological_df = ideological_df.dropna(subset=["stream_group"])

    all_streams_df = pd.concat([mixed_df, ideological_df], ignore_index=True)

    grouped = (
        all_streams_df.groupby(["stream_group", "time_window"])
        .agg(
            text=(TEXT_COLUMN, lambda texts: " ".join(texts)),
            n_articles=(TEXT_COLUMN, "count"),
            domains=(DOMAIN_COLUMN, lambda x: sorted(set(x))),
            labels=(LABEL_COLUMN, lambda x: sorted(set(x))),
        )
        .reset_index()
    )

    grouped = grouped[grouped["n_articles"] >= min_articles_per_window]
    grouped = grouped.sort_values(["stream_group", "time_window"]).reset_index(drop=True)

    return grouped

### Feature Extraction

In [472]:
def build_tfidf_features(window_df: pd.DataFrame):
    vectorizer = TfidfVectorizer(
        lowercase=True,
        stop_words="english",
        max_features=5000,
        min_df=2,
        max_df=0.9,
        ngram_range=(1, 2),
    )

    X = vectorizer.fit_transform(window_df["text"])
    return X.toarray(), vectorizer


def build_embedding_features(window_df: pd.DataFrame):
    from sentence_transformers import SentenceTransformer

    model = SentenceTransformer("all-MiniLM-L6-v2")

    X = model.encode(
        window_df["text"].tolist(),
        batch_size=16,
        show_progress_bar=True,
        normalize_embeddings=True,
    )

    return np.asarray(X), model

### Drift score stream

In [473]:
def compute_cosine_drift_scores(window_df: pd.DataFrame, X: np.ndarray) -> pd.DataFrame:
    rows = []

    temp = window_df.copy()
    temp["row_id"] = np.arange(len(temp))

    for stream_group, group_df in temp.groupby("stream_group"):
        group_df = group_df.sort_values("time_window")

        indices = group_df["row_id"].to_numpy()
        time_windows = group_df["time_window"].to_numpy()
        n_articles = group_df["n_articles"].to_numpy()

        if len(indices) < 2:
            continue

        for i in range(1, len(indices)):
            prev_idx = indices[i - 1]
            curr_idx = indices[i]

            prev_vec = X[prev_idx].reshape(1, -1)
            curr_vec = X[curr_idx].reshape(1, -1)

            score = cosine_distances(prev_vec, curr_vec)[0, 0]

            rows.append({
                "stream_group": stream_group,
                "time_window": pd.to_datetime(time_windows[i]),
                "drift_score": float(score),
                "n_articles": int(n_articles[i]),
            })

    return pd.DataFrame(rows)

### River detectors

In [474]:
def fresh_detectors():
    return {
        "ADWIN": drift.ADWIN(delta=0.1, clock=1),

        "PageHinkley": drift.PageHinkley(
            min_instances=3 if TIME_WINDOW == "D" else 2,
            delta=0.0005,
            threshold=3,
            alpha=0.99,
        ),

        "KSWIN": drift.KSWIN(
            alpha=0.05,
            window_size=8 if TIME_WINDOW == "D" else 5,
            stat_size=4 if TIME_WINDOW == "D" else 2,
        ),
    }


def is_drift_detected(detector) -> bool:
    if hasattr(detector, "drift_detected"):
        return detector.drift_detected

    if hasattr(detector, "change_detected"):
        return detector.change_detected

    return False


def run_detectors(score_df: pd.DataFrame) -> pd.DataFrame:
    detections = []

    for stream_group, group_df in score_df.groupby("stream_group"):
        group_df = group_df.sort_values("time_window")
        detectors = fresh_detectors()

        for _, row in group_df.iterrows():
            score = float(row["drift_score"])
            time_window = pd.to_datetime(row["time_window"])

            for detector_name, detector in detectors.items():
                detector.update(score)

                if is_drift_detected(detector):
                    detections.append({
                        "stream_group": stream_group,
                        "detector": detector_name,
                        "time_window": time_window,
                        "drift_score": score,
                        "n_articles": row["n_articles"],
                    })

                    print(
                        f"[DRIFT] stream={stream_group} | "
                        f"{detector_name} detected drift on "
                        f"{time_window.date()} | score={score:.4f}"
                    )

    return pd.DataFrame(
        detections,
        columns=[
            "stream_group",
            "detector",
            "time_window",
            "drift_score",
            "n_articles",
        ],
    )

### Events and evaluation

In [475]:
def prepare_events() -> pd.DataFrame:
    rows = []

    for event_name, date_str in EVENTS.items():
        event_date = pd.to_datetime(date_str)

        # Keep only events that fall inside the analyzed stream period.
        if event_date < START_DATE or event_date > END_DATE:
            continue

        if TIME_WINDOW == "D":
            event_window = event_date.floor("D")
        elif TIME_WINDOW == "W":
            event_window = event_date.to_period("W").start_time
        else:
            raise ValueError("TIME_WINDOW must be 'D' or 'W'")

        rows.append({
            "event": event_name,
            "event_date": event_date,
            "event_window": event_window,
        })

    return pd.DataFrame(rows)


def evaluate_detections(
    detections_df: pd.DataFrame,
    events_df: pd.DataFrame,
    tolerance_days: int = TOLERANCE_DAYS,
):
    if detections_df.empty:
        matched_cols = [
            "stream_group",
            "detector",
            "time_window",
            "drift_score",
            "matched_event",
            "delay_days",
            "is_true_positive",
        ]

        summary_cols = [
            "stream_group",
            "detector",
            "detections",
            "true_positives",
            "false_positives",
            "avg_delay_days",
            "events_detected",
            "events_missed",
        ]

        return pd.DataFrame(columns=matched_cols), pd.DataFrame(columns=summary_cols)

    tolerance = timedelta(days=tolerance_days)
    matched_rows = []

    for _, det in detections_df.iterrows():
        detection_window = pd.to_datetime(det["time_window"])

        matched_event = None
        min_delay_days = None

        for _, event in events_df.iterrows():
            event_date = pd.to_datetime(event["event_date"])
            diff = detection_window - event_date

            if abs(diff) <= tolerance:
                delay_days = diff.days

                if min_delay_days is None or abs(delay_days) < abs(min_delay_days):
                    matched_event = event["event"]
                    min_delay_days = delay_days

        matched_rows.append({
            "stream_group": det["stream_group"],
            "detector": det["detector"],
            "time_window": detection_window,
            "drift_score": det["drift_score"],
            "matched_event": matched_event,
            "delay_days": min_delay_days,
            "is_true_positive": matched_event is not None,
        })

    matched_df = pd.DataFrame(matched_rows)

    summary = (
        matched_df
        .groupby(["stream_group", "detector"])
        .agg(
            detections=("time_window", "count"),
            true_positives=("is_true_positive", "sum"),
            false_positives=("is_true_positive", lambda x: (~x).sum()),
            avg_delay_days=("delay_days", "mean"),
        )
        .reset_index()
    )

    event_coverage_rows = []

    for (stream_group, detector_name), sub in matched_df.groupby(["stream_group", "detector"]):
        detected_events = set(sub.dropna(subset=["matched_event"])["matched_event"])

        for event_name in events_df["event"]:
            event_coverage_rows.append({
                "stream_group": stream_group,
                "detector": detector_name,
                "event": event_name,
                "detected": event_name in detected_events,
            })

    coverage_df = pd.DataFrame(event_coverage_rows)

    if not coverage_df.empty:
        coverage_summary = (
            coverage_df
            .groupby(["stream_group", "detector"])
            .agg(
                events_detected=("detected", "sum"),
                events_missed=("detected", lambda x: (~x).sum()),
            )
            .reset_index()
        )

        summary = summary.merge(
            coverage_summary,
            on=["stream_group", "detector"],
            how="left",
        )

    return matched_df, summary

### Plots

In [476]:
def plot_drift_timeline(
    score_df: pd.DataFrame,
    detections_df: pd.DataFrame,
    events_df: pd.DataFrame,
    feature_type: str,
    time_window: str,
    stream_group: str,
    output_dir: Path,
):
    sub_scores = score_df[score_df["stream_group"] == stream_group].sort_values("time_window")

    if sub_scores.empty:
        print(f"No scores to plot for stream: {stream_group}")
        return

    w_name = "daily" if time_window == "D" else "weekly"

    plt.figure(figsize=(16, 7))

    plt.plot(
        sub_scores["time_window"],
        sub_scores["drift_score"],
        marker="o",
        linewidth=1.3,
        label=f"{stream_group} {w_name} cosine drift score",
    )

    max_score = sub_scores["drift_score"].max()

    # Known political event lines
    for _, event in events_df.iterrows():
        plt.axvline(
            event["event_date"],
            linestyle="-",
            linewidth=1.2,
            alpha=0.4,
            label=None,
        )

        plt.text(
            event["event_date"],
            max_score,
            event["event"],
            rotation=90,
            verticalalignment="top",
            fontsize=8,
        )

    # Detected drift lines: red vertical lines
    sub_det = detections_df[detections_df["stream_group"] == stream_group]

    if not sub_det.empty:
        first_drift_label_used = False

        for detector_name, sub in sub_det.groupby("detector"):
            plt.scatter(
                sub["time_window"],
                sub["drift_score"],
                s=80,
                label=f"{detector_name} alarm",
            )

            for _, row in sub.iterrows():
                plt.axvline(
                    x=row["time_window"],
                    color="red",
                    linestyle="--",
                    linewidth=1.3,
                    alpha=0.65,
                    label="Detected drift" if not first_drift_label_used else None,
                )
                first_drift_label_used = True

    plt.title(
        f"H1 distributional drift detection — {stream_group} stream — "
        f"{feature_type} — {w_name}"
    )
    plt.xlabel("Time")
    plt.ylabel("Cosine distance from previous window")
    plt.grid(True, linestyle=":", alpha=0.6)
    plt.legend(loc="upper left")
    plt.tight_layout()

    output_path = output_dir / "plot.png"
    plt.savefig(output_path, dpi=200)
    plt.close()

    print(f"Saved plot: {output_path}")


### Saving results

In [477]:
def save_grouped_results(
    score_df: pd.DataFrame,
    detections_df: pd.DataFrame,
    matched_df: pd.DataFrame,
    summary_df: pd.DataFrame,
    window_df: pd.DataFrame,
    events_df: pd.DataFrame,
):
    w_name = window_name()

    global_dir = get_global_output_dir(FEATURE_TYPE, TIME_WINDOW)

    window_df.drop(columns=["text"]).to_csv(global_dir / "windows.csv", index=False)
    score_df.to_csv(global_dir / "all_scores.csv", index=False)
    detections_df.to_csv(global_dir / "all_detections.csv", index=False)
    matched_df.to_csv(global_dir / "all_matched.csv", index=False)
    summary_df.to_csv(global_dir / "all_summary.csv", index=False)
    events_df.to_csv(global_dir / "events.csv", index=False)

    for stream_group in sorted(score_df["stream_group"].unique()):
        out_dir = get_output_dir(FEATURE_TYPE, TIME_WINDOW, stream_group)

        stream_scores = score_df[score_df["stream_group"] == stream_group]
        stream_detections = detections_df[detections_df["stream_group"] == stream_group]
        stream_matched = matched_df[matched_df["stream_group"] == stream_group]
        stream_summary = summary_df[summary_df["stream_group"] == stream_group]
        stream_windows = window_df[window_df["stream_group"] == stream_group].drop(columns=["text"])

        stream_windows.to_csv(out_dir / "windows.csv", index=False)
        stream_scores.to_csv(out_dir / "scores.csv", index=False)
        stream_detections.to_csv(out_dir / "detections.csv", index=False)
        stream_matched.to_csv(out_dir / "matched.csv", index=False)
        stream_summary.to_csv(out_dir / "summary.csv", index=False)

    print(f"Saved grouped results in: {OUTPUT_DIR / FEATURE_TYPE / w_name}")

### Main

In [478]:
def main():
    w_name = window_name()

    print("=" * 80)
    print("H1: Distributional political drift detection")
    print("=" * 80)

    print(f"\nInput file: {INPUT_FILE}")
    print(f"Feature type: {FEATURE_TYPE}")
    print(f"Time window: {TIME_WINDOW} ({w_name})")
    print(f"Min articles per window: {MIN_ARTICLES_PER_WINDOW}")
    print(f"Tolerance days: {TOLERANCE_DAYS}")
    print(f"Analysis date window: {START_DATE.date()} -> {END_DATE.date()}")

    df = load_jsonl(INPUT_FILE)

    print(f"\nLoaded articles after filtering: {len(df)}")
    print(f"Date range: {df['date'].min()} -> {df['date'].max()}")

    print("\nArticles per label:")
    print(df[LABEL_COLUMN].value_counts().sort_index())

    print("\nArticles per domain:")
    print(df[DOMAIN_COLUMN].value_counts())

    print(f"\nAggregating articles into {w_name} windows...")
    window_df = aggregate_stream_time_windows(df)

    if window_df.empty:
        raise ValueError(
            "No time windows left after filtering. "
            "Lower MIN_ARTICLES_PER_WINDOW or check your data."
        )

    print(f"\nNumber of stream windows: {len(window_df)}")
    print(window_df[["stream_group", "time_window", "n_articles", "domains", "labels"]].head())

    print(f"\nBuilding {FEATURE_TYPE} features...")

    if FEATURE_TYPE == "tfidf":
        X, _ = build_tfidf_features(window_df)
    elif FEATURE_TYPE == "embedding":
        X, _ = build_embedding_features(window_df)
    else:
        raise ValueError("FEATURE_TYPE must be 'tfidf' or 'embedding'")

    print(f"Feature matrix shape: {X.shape}")

    print("\nComputing drift scores...")
    score_df = compute_cosine_drift_scores(window_df, X)

    print("\nRunning River detectors...")
    detections_df = run_detectors(score_df)

    print("\nPreparing events...")
    events_df = prepare_events()

    print("\nEvaluating detections...")
    matched_df, summary_df = evaluate_detections(detections_df, events_df)

    print("\nSummary:")
    print(summary_df)

    print("\nSaving grouped results...")
    save_grouped_results(
        score_df=score_df,
        detections_df=detections_df,
        matched_df=matched_df,
        summary_df=summary_df,
        window_df=window_df,
        events_df=events_df,
    )

    print("\nCreating plots...")
    for stream_group in sorted(score_df["stream_group"].unique()):
        out_dir = get_output_dir(FEATURE_TYPE, TIME_WINDOW, stream_group)

        plot_drift_timeline(
            score_df=score_df,
            detections_df=detections_df,
            events_df=events_df,
            feature_type=FEATURE_TYPE,
            time_window=TIME_WINDOW,
            stream_group=stream_group,
            output_dir=out_dir,
        )

    print("\nDone.")


In [479]:
main()

H1: Distributional political drift detection

Input file: data\old\polarized_combined1.jsonl
Feature type: embedding
Time window: W (weekly)
Min articles per window: 15
Tolerance days: 14
Analysis date window: 2024-03-15 -> 2025-02-20

Loaded articles after filtering: 2021
Date range: 2024-03-15 18:45:00 -> 2025-02-19 22:15:00

Articles per label:
label
0     812
1    1209
Name: count, dtype: int64

Articles per domain:
domain
breitbart.com      1209
thenation.com       568
motherjones.com     244
Name: count, dtype: int64

Aggregating articles into weekly windows...

Number of stream windows: 102
  stream_group time_window  n_articles                           domains  \
0         left  2024-04-01          21  [motherjones.com, thenation.com]   
1         left  2024-04-08          19  [motherjones.com, thenation.com]   
2         left  2024-04-15          17  [motherjones.com, thenation.com]   
3         left  2024-04-22          16  [motherjones.com, thenation.com]   
4         left 

Batches: 100%|██████████| 7/7 [00:06<00:00,  1.02it/s]


Feature matrix shape: (102, 384)

Computing drift scores...

Running River detectors...

Preparing events...

Evaluating detections...

Summary:
Empty DataFrame
Columns: [stream_group, detector, detections, true_positives, false_positives, avg_delay_days, events_detected, events_missed]
Index: []

Saving grouped results...
Saved grouped results in: results\h1\embedding\weekly

Creating plots...
Saved plot: results\h1\embedding\weekly\left\plot.png
Saved plot: results\h1\embedding\weekly\mixed\plot.png
Saved plot: results\h1\embedding\weekly\right\plot.png

Done.
